In [ ]:
import pandas as pd
import numpy as np
import json
import re, pytz, os, requests, sys
from pathlib import Path
from datetime import datetime
import sys
sys.path.append("/workspaces/service-data")

from src.clean import clean_percentage, clean_fiscal_yr, normalize_string, standardize_column_names
from src.load import load_csv
from src.export import export_to_csv
from src.merge import merge_si, merge_ss
from src.utils import dept_list, program_list
from main import get_config

import pandas as pd
import numpy as np
import pytz
from pathlib import Path

base_dir = Path.cwd()
parent_dir = base_dir.parent
config = get_config()
# dept = dept_list(config)

# si_url = https://github.com/gcperformance/service-data/releases/latest/download/si.csv
# si = pd.read_csv(si_url, keep_default_na=False, na_values='', delimiter=';')

# si_path = parent_dir / 'outputs' / 'si.csv'
# si = pd.read_csv(si_path, keep_default_na=False, na_values='', delimiter=';', engine='python', skipfooter=2)

# ss_path = parent_dir / 'outputs' / 'ss.csv'
# ss = pd.read_csv(ss_path, keep_default_na=False, na_values='', delimiter=';', engine='python', skipfooter=2)

timezone = pytz.timezone('America/Montreal')
current_datetime = pd.Timestamp.now(tz=timezone)
current_datetime_str = current_datetime.strftime("%Y-%m-%d_%H:%M:%S")

In [ ]:
# local_and_live_paths = [
#     ['si_all', parent_dir / 'outputs' / 'utils' / 'si_all.csv', 'https://github.com/gcperformance/service-data/releases/latest/download/si_all.csv'],
#     ['ss_all', parent_dir / 'outputs' / 'utils' / 'ss_all.csv', 'https://github.com/gcperformance/service-data/releases/latest/download/ss_all.csv'],
#     ['drf', parent_dir / 'outputs' / 'utils' / 'drf.csv', 'https://github.com/gcperformance/service-data/releases/latest/download/drf.csv'],
#     ['program_list', parent_dir / 'outputs' / 'utils' / 'program_list.csv', 'https://github.com/gcperformance/service-data/releases/latest/download/program_list.csv'],
# ]

# # Dataframe aligning service standard channels to service inventory volume channels
# local_and_live_paths = pd.DataFrame(local_and_live_paths, columns=['table', 'local_path', 'live_url'])

local_and_live_paths = config['local_and_live_paths']
local_and_live_paths = pd.DataFrame(local_and_live_paths, columns=['table', 'local_path', 'live_url'])

for table in local_and_live_paths['table']:
    # print(local_and_live_paths['live_url'][local_and_live_paths['table']==table]) 

    local = pd.DataFrame()
    live = pd.DataFrame()

    local_path = local_and_live_paths.loc[local_and_live_paths['table']==table, 'local_path'].item()
    local = pd.read_csv(local_path, dtype=str, keep_default_na=False, delimiter=';', engine='python', skipfooter=2)

    live_url =  local_and_live_paths.loc[local_and_live_paths['table']==table, 'live_url'].item()
    live = pd.read_csv(live_url, dtype=str, keep_default_na=False, delimiter=';', engine='python', skipfooter=2)

    check = live.ne(local).melt()
    publish_flag = not (check[check['value']].shape[0] > 0)

    flag_path = Path.cwd() / "publish.flag"
    if publish_flag:
        flag_path.write_text(f"Ok to publish - Timestamp:{current_datetime_str}\n", encoding="utf-8")
        break
    else:
        flag_path.unlink(missing_ok=True)



In [ ]:
testing_break_list = [0,1,2,3,4,5,6,7,8,'poop',9,1,2,3]

for item in testing_break_list:
    if item == 'poop':
        break
    print(item)